In [ ]:
import os
import re
from io import StringIO
from pathlib import Path
import gc 

import anndata as ad
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline
import math

import cell2location
from cell2location.utils import select_slide
from cell2location.plt import plot_spatial
from cell2location.utils.filtering import filter_genes
from cell2location.models import RegressionModel, Cell2location
import torch 

import squidpy as sq # optional I think
mpl.rcParams['figure.dpi'] = 200

In [ ]:
results_folder = "../../../data/cell2location/"
ref_run_name = os.path.join(results_folder, "reference_signatures")
run_name = os.path.join(results_folder, "cell2location_map")

# Set up reference signatures

## Run regression model

In [ ]:
adata_ref = sc.read_h5ad('../../../data/GBM_LEAP_annotations/GBM_LEAP_annotations_v2.10_11_23.h5ad')

In [ ]:
donors = sorted(set(adata_refs.obs['donor_id']))

adata_refs = {}
for batch in donors:

    adata_ref_batch = adata_ref[adata_ref.obs['donor_id']==batch].copy()

    ## Filter genes on the basis patient expression
    selected = filter_genes(adata_ref_batch, cell_count_cutoff=100, 
                            cell_percentage_cutoff2=0.5, nonz_mean_cutoff=1.1)

    # prepare anndata for regression model 
    adata_ref_batch = adata_ref_batch[:, selected].copy()
    RegressionModel.setup_anndata(adata=adata_ref_batch,
                            batch_key='sample',
                            labels_key='TME_GBM_granular'
                           )
    mod = RegressionModel(adata_ref_batch)
    mod.view_anndata_setup(adata_ref_batch)

    mod.train(max_epochs=400, batch_size=10000, train_size=1, lr=0.002, use_gpu=True)
    mod.plot_history(20)
    plt.savefig('{}_regression_training.png'.format(batch))

    mod.save(ref_run_name, overwrite=True)
    adata_ref_batch = mod.export_posterior(
        adata_ref_batch, sample_kwargs={'num_samples': 10000, 'batch_size': 10000, 'use_gpu': True}
    )
    adata_ref_batch.write_h5ad(os.path.join(ref_run_name, "run_sc_{}.h5ad".format(batch)))

    adata_refs[batch] = adata_ref_batch
    torch.cuda.empty_cache()
    gc.collect()

# Core cell2location 

In [ ]:
# Confirm ref dir:
ref_dir =  "../../../data/cell2location/reference_signatures/"

## Load sample lists

- Consistent directory structure so the sample manifest I use for cell2loc is based off of this
- I generally load a pre-existing csv that I've made
- Might not be relevant but this is (generally) how I generate the sample list files

In [ ]:
# cytassist
sample_directory = '../../../data/cytassist/phase_1'
samples = [folder for folder, subs, files in os.walk(sample_directory)\
    if folder.endswith('GRCh38-2020-A')]
samples = [i for i in samples if 'AT20' not in i]
samples = [i for i in samples if 'unknown' not in i]

sample_list_cytassist = []
for path in samples:
    site_id = re.sub('^.*/(.*?)/spac.*$', '\\1', path)
    donor_id = re.sub('^.*phase_1/(.*?)/.*$', '\\1', path)
    sample = re.sub('^.*/', '', path)
    

    
    sample_list_cytassist.append(pd.DataFrame({'sample':sample,
                                     'site_id':site_id,
                                     'sample_name':site_id, # to make downstream analyses simple
                                     'donor_id':donor_id,
                                     'paths':path}, index=[0]))
sample_list_cytassist = pd.concat(sample_list_cytassist, ignore_index=True)

In [ ]:
# Visium (standard)
sample_directory = '../../../data/visium_data/'
samples = [folder for folder, subs, files in os.walk(sample_directory)\
    if folder.endswith('GRCh38-2020-A')]
samples = [i for i in samples if 'AT20' not in i]
samples = [i for i in samples if 'unknown' not in i]

sample_list = []
for path in samples:
    site_id = re.sub('^.*/(.*?)/spac.*$', '\\1', path)
    donor_id = re.sub('^.*phase_1/(.*?)/.*$', '\\1', path)
    sample = re.sub('^.*/', '', path)
    

    
    sample_list.append(pd.DataFrame({'sample':sample,
                                     'site_id':site_id,
                                     'sample_name':site_id, # to make downstream analyses simple
                                     'donor_id':donor_id,
                                     'paths':path}, index=[0]))
sample_list = pd.concat(sample_list, ignore_index=True)

**Dataframe structure**:

In [ ]:
# sample	site_id	donor_id	paths	sample_name
#0	spaceranger200_count_45730_GBM_SPA13091863_GRC...	AT10-BRA-5-FO-1	AT10	visium_data/AT10-BRA-5-F...	AT10-BRA-5-FO-1_1
#1	spaceranger200_count_45730_GBM_SPA13091864_GRC...	AT10-BRA-5-FO-1	AT10	visium_data/AT10-BRA-5-F...	AT10-BRA-5-FO-1_2
#2	spaceranger200_count_45730_GBM_SPA13091855_GRC...	AT10-BRA-5-FO-2	AT10	visium_data/AT10-BRA-5-F...	AT10-BRA-5-FO-2_1

## Load cell2loc

In [ ]:
batch_adata_vis = {}
for batch in set(sample_list['donor_id']):
    
    batch_samples = sample_list[sample_list['sample_name'].str.contains(batch)]
    
    # Load adatas for batch and generate basic QC metrics
    adatas = {}
    for sample in batch_samples['sample']:
            sample_name =  batch_samples.loc[batch_samples['sample']==sample]['sample_name'].item()
            site_id = batch_samples.loc[batch_samples['sample']==sample]['site_id'].item()
            path = batch_samples.loc[batch_samples['sample']==sample]['paths'].item()
            
            
            # Pre-processing steps
            adata = sq.read.visium(path, library_id=sample_name)
            adata.var = (adata.var.reset_index().
                    rename(columns={"index": "SYMBOL", "gene_ids": "ENSEMBL"}).
                    set_index("ENSEMBL"))
            adata.obs['sample'] = sample
            adata.obs['sample_name'] = sample_name
            adata.var['mt'] = [gene.startswith('MT-') for gene in adata.var['SYMBOL']]
            sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
            adata.uns["sample_name"] = sample_name
            adatas[sample] = adata
    adata_vis = ad.concat(adatas, label="sample", merge="same", 
                      uns_merge="unique", index_unique="_")
        
    adata_vis.obsm['mt'] = adata_vis.X[:, adata_vis.var['mt'].to_numpy()]
    adata_vis = adata_vis[:, ~adata_vis.var['mt'].to_numpy()]
    adata_vis.var = adata_vis.var.reset_index()
    adata_vis.var_names = adata_vis.var['SYMBOL']
    adata_vis = adata_vis[: , ~adata_vis.var['SYMBOL'].duplicated(keep='first')]
        
    batch_adata_vis[batch] = adata_vis

## Run cell2loc

In [ ]:
# batches = whichever batches you need to run (example here: ['AT15', 'AT3'...]

for donor in batches:
    
    print('Load ref adata... {}'.format(donor))
    # Load ref sigs from previous step (per donor)
    adata_ref = ad.read_h5ad(os.path.join(ref_run_name, "run_sc_{}.h5ad".format(donor)))
    adata_vis = batch_adata_vis[donor]

    print('Setup adatas... {}'.format(donor))
    # Set up signatures
    inf_aver = adata_ref.varm['q05_per_cluster_mu_fg'][[f'q05_per_cluster_mu_fg_{i}'
                                        for i in adata_ref.uns['mod']['factor_names']]]
    inf_aver.columns = adata_ref.uns['mod']['factor_names']
    
    intersect = adata_ref.var_names.intersection(adata_vis.var_names)

    adata_vis = adata_vis[:, intersect].copy()
    inf_aver = inf_aver.loc[intersect, :]

    # set up model
    # might be good to assess parameters here in a few demo runs
    Cell2location.setup_anndata(adata_vis, batch_key="sample")
    mod = cell2location.models.Cell2location(
        adata_vis, cell_state_df=inf_aver,
        N_cells_per_location=30,
        detection_alpha=200)
    mod.view_anndata_setup(adata_vis)
        
    # train model
    # added batch size arg; necessary for cytassist or large batches
    mod.train(max_epochs=6000,
              batch_size=math.ceil((1/4) * adata_vis.n_obs),
              train_size=1)

    mod.plot_history()
    plt.legend(labels=['full data training'])
    plt.savefig('epoch_QC_batch{}'.format(donor))

    # Again, memory issues means this step is potentially problematic; should test
    # 'use_quantiles = True' exports less information but won't cause a crash 
    # and should still provide interpretable abundance estimates
    mod.save('{}/{}'.format(run_name, donor), overwrite=True)
    adata_vis = mod.export_posterior(
                adata_vis, use_quantiles = True)
    adata_vis.write_h5ad(os.path.join(run_name, "sp_batch{}.h5ad".format(donor)))

    # Important to note: delete model object as it occupies gpu mem
    del(mod)
    torch.cuda.empty_cache()
    del(adata_ref)
    gc.collect()

Load ref adata... AT15
Setup adatas... AT15


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Anndata setup with scvi-tools version 1.1.2.

Setup via `Cell2location.setup_anndata` with arguments:

{
│   'layer': None,
│   'batch_key': 'sample',
│   'labels_key': None,
│   'categorical_covariate_keys': None,
│   'continuous_covariate_keys': None
}

...

               Data Registry                
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Registry Key ┃    scvi-tools Location    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      X       │          adata.X          │
│    batch     │ adata.obs['_scvi_batch']  │
│    ind_x     │   adata.obs['_indices']   │
│    labels    │ adata.obs['_scvi_labels'] │
└──────────────┴───────────────────────────┘

                                          batch State Registry                                          
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃   Source Location   ┃                        Categories                        ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['sample'] │ spaceranger210_count_47450_GBM_SPA13740726_GRCh38-2020-A │          0          │
│                     │ spaceranger210_count_47450_GBM_SPA13740728_GRCh38-2020-A │          1          │
│                     │ spaceranger210_count_47450_GBM_SPA13740723_GRCh38-2020-A │          2          │
│                     │ spaceranger210_count_47450_GBM_SPA13740725_GRCh38-2020-A │          3          │
│                     │ spaceranger210_count_46937_GBM_SPA13635043_GRCh38-2020-A │          4          │
│                     │ spaceranger210_count_47450_GBM_SPA13740724_GRCh38-2020-A │          5          │
│                     │ spaceranger210_count_47268_GBM_SPA13669471_GRCh38-2020-A │          6          │
│                     │ spaceranger210_count_47268_GBM_SPA13669470_GRCh38-2020-A │          7          │
│                     │ spaceranger210_count_47450_GBM_SPA13740727_GRCh38-2020-A │          8          │
└─────────────────────┴──────────────────────────────────────────────────────────┴─────────────────────┘

                     labels State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['_scvi_labels'] │     0      │          0          │
└───────────────────────────┴────────────┴─────────────────────┘

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/nfs/team283/gd11/software/miniconda3/envs/cell2loc_env/lib/python3.10/site-packages/lightning/pytorch/trainer/configuration_validator.py:72: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/nfs/team283/gd11/software/miniconda3/envs/cell2loc_env/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:293: The number of training batches (4) is smaller than the logging interval Trainer(log_every_n_s

Epoch 5779/6000:  96%|▉| 5778/6000 [7:23:31<17:01,  4.60s/it, v_num=1, elbo_train=1.21e+